# Diabetes Prediction (Advanced Optimization)

This notebook implements an advanced diabetes prediction pipeline featuring:
1.  **KNN Imputation**: For more accurate missing value estimation.
2.  **Feature Engineering**: Categorizing BMI to capture non-linear risks.
3.  **Outlier Capping**: Winsorization to handle extreme values.
4.  **Ensemble Learning**: A Voting Classifier combining Random Forest, Gradient Boosting, and Logistic Regression.

In [92]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.impute import KNNImputer
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import pickle
import warnings
warnings.filterwarnings('ignore')

## 1. Load Data & Advanced Preprocessing

In [93]:
df = pd.read_csv('datasets/diabetes.csv')

# 1. Handle 0 values (Replace with NaN)
cols_to_fix = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
df[cols_to_fix] = df[cols_to_fix].replace(0, np.nan)

# 2. KNN Imputation (More robust than median)
knn_imputer = KNNImputer(n_neighbors=5)
df[cols_to_fix] = knn_imputer.fit_transform(df[cols_to_fix])

# 3. Outlier Capping (Winsorization) - Retained from previous success
def cap_outliers(df, col):
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    low_lim = Q1 - 1.5 * IQR
    up_lim = Q3 + 1.5 * IQR
    df[col] = np.where(df[col] < low_lim, low_lim, df[col])
    df[col] = np.where(df[col] > up_lim, up_lim, df[col])
    return df

for col in ['Insulin', 'DiabetesPedigreeFunction']:
    df = cap_outliers(df, col)

# 4. Feature Engineering: BMI Categories
# Underweight (<18.5), Normal (18.5-24.9), Overweight (25-29.9), Obese (>30)
# We encode this as an ordinal feature: 0, 1, 2, 3
def categorize_bmi(bmi):
    if bmi < 18.5: return 0
    elif 18.5 <= bmi < 25: return 1
    elif 25 <= bmi < 30: return 2
    else: return 3

df['BMI_Cat'] = df['BMI'].apply(categorize_bmi)

X = df.drop('Outcome', axis=1)
y = df['Outcome']

# Scale
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

## 2. Model Training: Ensemble Voting Classifier
Combining Random Forest (Tuned), Gradient Boosting, and Logistic Regression.

In [94]:
# Base Models
rf = RandomForestClassifier(n_estimators=20, max_depth=20, min_samples_split=5, min_samples_leaf=1, random_state=42)
gb = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42)
lr = LogisticRegression(solver='liblinear', random_state=42)

# Voting Classifier (Soft Voting for probability averaging)
voting_clf = VotingClassifier(
    estimators=[
        ('rf', rf),
        ('gb', gb),
        ('lr', lr)
    ],
    voting='soft'
)

voting_clf.fit(X_train, y_train)

,"estimators estimators: list of (str, estimator) tuplesInvoking the ``fit`` method on the ``VotingClassifier`` will fit clonesof those original estimators that will be stored in the class attribute``self.estimators_``. An estimator can be set to ``'drop'`` using:meth:`set_params`... versionchanged:: 0.21 ``'drop'`` is accepted. Using None was deprecated in 0.22 and support was removed in 0.24.","[('rf', ...), ('gb', ...), ...]"
,"voting voting: {'hard', 'soft'}, default='hard'If 'hard', uses predicted class labels for majority rule voting.Else if 'soft', predicts the class label based on the argmax ofthe sums of the predicted probabilities, which is recommended foran ensemble of well-calibrated classifiers.",'soft'
,"weights weights: array-like of shape (n_classifiers,), default=NoneSequence of weights (`float` or `int`) to weight the occurrences ofpredicted class labels (`hard` voting) or class probabilitiesbefore averaging (`soft` voting). Uses uniform weights if `None`.",None
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel for ``fit``.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionadded:: 0.18",None
,"flatten_transform flatten_transform: bool, default=TrueAffects shape of transform output only when voting='soft'If voting='soft' and flatten_transform=True, transform method returnsmatrix with shape (n_samples, n_classifiers * n_classes). Ifflatten_transform=False, it returns(n_classifiers, n_samples, n_classes).",True
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting will be printed as itis completed... versionadded:: 0.23",False
,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",20
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",20
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",5
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1


## 3. Final Evaluation

In [95]:
y_train_pred = voting_clf.predict(X_train)
y_test_pred = voting_clf.predict(X_test)

train_acc = accuracy_score(y_train, y_train_pred)
test_acc = accuracy_score(y_test, y_test_pred)

print(f"Training Accuracy: {train_acc:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")

print("\nClassification Report (Test):")
print(classification_report(y_test, y_test_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_test_pred))

Training Accuracy: 0.9397
Test Accuracy: 0.7662

Classification Report (Test):
              precision    recall  f1-score   support

           0       0.81      0.83      0.82        99
           1       0.68      0.65      0.67        55

    accuracy                           0.77       154
   macro avg       0.75      0.74      0.74       154
weighted avg       0.76      0.77      0.77       154


Confusion Matrix:
[[82 17]
 [19 36]]


## 4. Save Best Model

In [96]:
pickle.dump(voting_clf, open('../models/diabetes_model.pkl', 'wb'))
pickle.dump(scaler, open('../models/diabetes_scaler.pkl', 'wb'))
print("Optimized Voting Classifier model saved as diabetes_model.pkl")

Optimized Voting Classifier model saved as diabetes_model.pkl


## 5. Verification
Load the saved model and test on a random sample from the test set.

In [98]:
# Verification
import pickle

# Load model
model = pickle.load(open('diabetes_model.pkl', 'rb'))

# Test on a sample from X_test (assuming X_test is in memory from execution)
# if X_test is not found, please run the cells above first to generate the test set.
try:
    sample_idx = 0
    sample_data = X_test[sample_idx].reshape(1, -1)
    prediction = model.predict(sample_data)

    print(f"Predicted Class: {prediction[0]}")
    # Assuming y_test is a Series or array
    actual = y_test.iloc[sample_idx] if hasattr(y_test, 'iloc') else y_test[sample_idx] if isinstance(y_test, (pd.Series, np.ndarray)) else y_test
    print(f"Actual Class: {actual}")
except NameError:
    print("X_test not found. Please run the notebook cells above to load data first.")

Predicted Class: 0
Actual Class: 0
